# 09 — Corpus-Level Corroboration (Tables 13, 15, 17)

Everything so far rests on two encoder-derived quantities, TP and IP. If the
story is right, it should also show up in measures that never touch a model
vocabulary. This notebook computes three such checks.

| Measure | What it is | Why it corroborates |
|---|---|---|
| **MATTR** (w = 500) | Moving-average type-token ratio over whitespace tokens | Vocabulary-free morphological diversity. Reproduces the TP/IP hierarchy (MAL > TAM > MAR > GUJ > HIN) without reference to any tokenizer |
| **Word → token Spearman** | Rank correlation of word count against XLM-R token count | Romanisation pushes it toward 1.0 — the signature of single-character BPE fallback |
| **IP–COMET Pearson** | Sentence-level, native script | Links the information-density measure directly to the score under audit |

**Input:** `../data/indic/indic_parity_xlmr.xlsx`
**Output:** `../results/tables/corpus_corroboration.csv`

## Setup

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd

# ── Paths (relative; nothing in this repository uses an absolute path) ───────
DATA_XLMR   = Path("../data/indic/indic_parity_xlmr.xlsx")
DATA_MULTI  = Path("../data/indic/indic_parity_multi_tokenizer.xlsx")
DATA_LATIN  = Path("../data/latin/wmt24_ende_enes_metrics.xlsx")
TABLES_DIR  = Path("../results/tables")
FIGURES_DIR = Path("../results/figures")
TABLES_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

# ── Sheet names exactly as they appear in the workbook ───────────────────────
SHEET_MAP = {
    "GUJ": "Indic_mt _for_analysis - Gujara",
    "TAM": "Indic_mt _for_analysis - Tamil_",
    "MAL": "Indic_mt _for_analysis - Malaya",
    "MAR": "Indic_mt _for_analysis - Marath",
    "HIN": "Indic_mt _for_analysis - Hindi_",
}

# ── Language display order (fixed throughout the paper) ──────────────────────
LANG_ORDER = ["GUJ", "TAM", "MAL", "MAR", "HIN"]

# ── Column names (as in the xlsx) ────────────────────────────────────────────
COL_COMET_NAT = "COMET"                                          # native-script COMET
COL_COMET_ROM = "COMET_romanized"                                # romanised COMET
COL_TP_NAT    = "Translation_xlmr_TP"                            # TP, native
COL_TP_ROM    = "Translation_Transliteration_romanized_xlmr_TP"  # TP, romanised
COL_IP_NAT    = "Translation_xlmr_IP"                            # IP, native
COL_IP_ROM    = "Translation_Transliteration_romanized_xlmr_IP"  # IP, romanised
COL_HUMAN     = "Human_scores"                                   # MQM-derived human score
COL_SEVERITY  = "Error1_Severity"                                # primary error severity

# ── Seeds (every stochastic step in this repository) ─────────────────────────
SEED_SPLIT = 42   # 50/50 within-language train/test split
SEED_GBM   = 0    # GradientBoostingRegressor
SEED_PERM  = 0    # paired permutation test

print("Config loaded. DATA_XLMR:", DATA_XLMR)

## Loading the Language Sheets

In [ ]:
def load_sheets(path):
    """Read the five per-language sheets.

    Returns two dicts keyed by ISO code:
      full  — all 1,400 rows per language (the 7,000-segment base)
      work  — rows carrying a numeric human score (the 6,995-segment base)

    Coercing the human-score column to numeric is what removes the five
    unusable rows: four are blank and one (Malayalam) holds the string
    ``\`19``, which is not a score.
    """
    full, work = {}, {}
    for lang in LANG_ORDER:
        d = pd.read_excel(path, sheet_name=SHEET_MAP[lang])
        d["H"] = pd.to_numeric(d[COL_HUMAN], errors="coerce")
        full[lang] = d
        work[lang] = d.dropna(subset=["H"]).reset_index(drop=True)
    return full, work


full, work = load_sheets(DATA_XLMR)
print(f"Loaded {sum(len(full[l]) for l in LANG_ORDER):,} rows "
      f"across {len(SHEET_MAP)} sheets")

from scipy import stats

## MATTR (w = 500)

Computed over whitespace tokens on the concatenated target text, native and
romanised. MATTR is order-dependent, so the sheet order is kept as committed.

In [ ]:
def mattr(texts, window=500):
    """Moving-average type-token ratio (Covington & McFall, 2010)."""
    tokens = []
    for t in texts:
        tokens.extend(str(t).split())
    if len(tokens) < window:
        return len(set(tokens)) / max(len(tokens), 1)
    return float(np.mean([len(set(tokens[i:i + window])) / window
                          for i in range(len(tokens) - window + 1)]))


print("MATTR, native \u2192 romanised")
print(f"{'Lang':>5}  {'native':>7}  {'romanised':>10}  {'\u0394':>7}")
print("-" * 34)
mattr_nat, mattr_rom = {}, {}
for lang in LANG_ORDER:
    d = full[lang]
    mattr_nat[lang] = mattr(d["Translation"].tolist())
    mattr_rom[lang] = mattr(d["Translation_Transliteration_romanized"].tolist())
    print(f"{lang:>5}  {mattr_nat[lang]:>7.3f}  {mattr_rom[lang]:>10.3f}  "
          f"{mattr_rom[lang] - mattr_nat[lang]:>+7.3f}")

# ── Cross-verification against Table 16 ──────────────────────────────────────
PAPER_MATTR = {"GUJ": (0.752, 0.665), "TAM": (0.862, 0.733), "MAL": (0.897, 0.768),
               "MAR": (0.827, 0.721), "HIN": (0.633, 0.601)}
for lang, (a, b) in PAPER_MATTR.items():
    assert abs(mattr_nat[lang] - a) < 0.001, f"{lang} MATTR nat"
    assert abs(mattr_rom[lang] - b) < 0.001, f"{lang} MATTR rom"
assert all(mattr_rom[l] < mattr_nat[l] for l in LANG_ORDER)
print("\n\u2713 All ten MATTR values match Table 17")
print("\u2713 MATTR falls under romanisation for every language")

order = sorted(LANG_ORDER, key=lambda l: -mattr_nat[l])
print(f"  Native MATTR ordering: {' > '.join(order)}  (paper: MAL > TAM > MAR > GUJ > HIN)")

## Word Count vs XLM-R Token Count

If romanisation were merely re-spelling, this correlation would be stable. It
rises toward 1.0 in every language instead: once the target is in Latin script
the tokenizer falls back to near-character-level pieces, so token count becomes
an almost deterministic function of word count.

In [ ]:
print("Sentence-level Spearman \u03c1: word count vs XLM-R token count (N = 1,400)")
print(f"{'Lang':>5}  {'native':>7}  {'romanised':>10}  {'\u0394':>7}")
print("-" * 34)
wt_nat, wt_rom = {}, {}
for lang in LANG_ORDER:
    d = full[lang]
    wn = d["Translation"].astype(str).str.split().str.len()
    wr = d["Translation_Transliteration_romanized"].astype(str).str.split().str.len()
    wt_nat[lang] = stats.spearmanr(wn, d["Translation_xlmr_token_count"])[0]
    wt_rom[lang] = stats.spearmanr(
        wr, d["Translation_Transliteration_romanized_xlmr_token_count"])[0]
    print(f"{lang:>5}  {wt_nat[lang]:>7.3f}  {wt_rom[lang]:>10.3f}  "
          f"{wt_rom[lang] - wt_nat[lang]:>+7.3f}")

PAPER_T14 = {"GUJ": (0.883, 0.952), "TAM": (0.850, 0.905), "MAL": (0.875, 0.900),
             "MAR": (0.891, 0.942), "HIN": (0.897, 0.956)}
for lang, (a, b) in PAPER_T14.items():
    assert abs(wt_nat[lang] - a) < 0.001, f"{lang} word-token nat"
    assert abs(wt_rom[lang] - b) < 0.001, f"{lang} word-token rom"
assert all(wt_rom[l] > wt_nat[l] for l in LANG_ORDER)
print("\n\u2713 All ten values match Table 15")
print("\u2713 Romanisation pushes every correlation toward 1.0 "
      "(single-character BPE fallback)")

## Sentence-Level IP–COMET Correlation

Native script, all 1,400 segments per language (no human score required).

In [ ]:
print(f"{'Lang':>5}  {'Pearson r':>10}  {'p':>10}")
print("-" * 29)
ipc = {}
for lang in LANG_ORDER:
    d = full[lang]
    r, p = stats.pearsonr(d[COL_IP_NAT], d[COL_COMET_NAT])
    ipc[lang] = r
    print(f"{lang:>5}  {r:>+10.3f}  {p:>10.3f}")

PAPER_IPC = {"GUJ": -0.064, "TAM": -0.263, "MAL": -0.266, "MAR": -0.216, "HIN": 0.203}
for lang, v in PAPER_IPC.items():
    assert abs(ipc[lang] - v) < 0.001, f"{lang} IP-COMET r"
print("\n\u2713 All five match Table 13")
print("  HIN is the sole positive coefficient; GUJ is weak but significant "
      "(r = -0.064, p = 0.017).")

## Saving Results

In [ ]:
out = pd.DataFrame({
    "mattr_nat": mattr_nat, "mattr_rom": mattr_rom,
    "word_token_rho_nat": wt_nat, "word_token_rho_rom": wt_rom,
    "ip_comet_r": ipc,
}).loc[LANG_ORDER]
path = TABLES_DIR / "corpus_corroboration.csv"
out.to_csv(path)
print(out.round(3).to_string())
print(f"\nSaved \u2192 {path}")
print("\n=== Notebook 09 — output manifest ===")
print("  corpus_corroboration.csv")

## Byte Premium (Table 17)

UTF-8 bytes per word in the target divided by bytes per word in the English
source. The mean is taken over **individual words**, not over sentences: a
per-sentence bytes/words ratio gives 2.75 for Gujarati where the reported figure
is 3.03.

Byte premium is vocabulary-free and encoding-only. It sits between 2.45x and
5.33x for every Indic language, which is exactly why it cannot explain the
ordering among them: the TP/IP hierarchy is not a UTF-8 artefact.

In [ ]:
def bytes_per_word(texts):
    # Mean UTF-8 byte length over individual whitespace-delimited words.
    lengths = [len(w.encode("utf-8")) for t in texts if pd.notna(t)
               for w in str(t).split()]
    return float(np.mean(lengths))


print("Byte premium, target vs English source")
print(f"{'Lang':>5}  {'bpw source':>11}  {'bpw target':>11}  {'premium':>8}")
print("-" * 42)
byte_prem = {}
for lang in LANG_ORDER:
    d = full[lang]
    src, tgt = bytes_per_word(d["Source"]), bytes_per_word(d["Translation"])
    byte_prem[lang] = tgt / src
    print(f"{lang:>5}  {src:>11.3f}  {tgt:>11.3f}  {tgt / src:>8.3f}")

PAPER_BP = {"GUJ": 3.03, "TAM": 5.30, "MAL": 5.33, "MAR": 3.64, "HIN": 2.45}
for lang, v in PAPER_BP.items():
    assert abs(byte_prem[lang] - v) < 0.005, f"{lang} byte premium"
print("\n\u2713 All five match Table 17")

## Isolated Dependent Vowels (Table 13)

In Indic abugidas a dependent vowel (matra) is a diacritic belonging to the
preceding consonant. A well-behaved tokenizer keeps the consonant-vowel unit
together; XLM-R's BPE frequently emits the diacritic as a standalone token
carrying no meaning on its own.

A token counts as isolated when it is **exactly one character** drawn from the
script's dependent-vowel set. The single-character condition matters: counting
any token composed of combining marks over-counts Hindi by roughly 1.8x.

In [ ]:
DEP_VOWELS = {
    "Devanagari": set("ािीुूृॄे"
                      "ैोौ्ंः"),
    "Gujarati": set("ાિીુૂૃેૈ"
                    "ોૌ્ંઃ"),
    "Tamil": set("ாிீுூெேை"
                 "ொோௌ்ஂ"),
    "Malayalam": set("ാിീുൂൃേൈ"
                     "ൊോൌ്ംഃ഻഼"),
}
SCRIPT_OF = {"GUJ": "Gujarati", "TAM": "Tamil", "MAL": "Malayalam",
             "MAR": "Devanagari", "HIN": "Devanagari"}


def dep_vowel_rate(token_series, script):
    # Isolated dependent-vowel tokens per 1,000 XLM-R tokens.
    charset = DEP_VOWELS[script]
    total = isolated = 0
    for s in token_series.astype(str):
        for tok in (t.strip() for t in s.split("|")):
            if not tok:
                continue
            total += 1
            if len(tok) == 1 and tok in charset:
                isolated += 1
    return isolated, total, isolated / total * 1000


print("Isolated dependent-vowel tokens, native script")
print(f"{'Lang':>5}  {'script':>11}  {'isolated':>9}  {'tokens':>8}  {'per 1k':>8}")
print("-" * 50)
dv = {}
for lang in LANG_ORDER:
    iso, tot, rate = dep_vowel_rate(full[lang]["Translation_xlmr_tokens"],
                                    SCRIPT_OF[lang])
    dv[lang] = rate
    print(f"{lang:>5}  {SCRIPT_OF[lang]:>11}  {iso:>9,}  {tot:>8,}  {rate:>8.3f}")

assert abs(dv["GUJ"] - 36.7) < 0.05
assert abs(dv["TAM"] - 13.6) < 0.05
assert abs(dv["MAL"] - 42.5) < 0.05
print(f"\n\u2713 GUJ {dv['GUJ']:.1f}, TAM {dv['TAM']:.1f}, "
      f"MAL {dv['MAL']:.1f} match Table 13")
print(f"\u2713 MAR {dv['MAR']:.1f}, HIN {dv['HIN']:.1f}")
assert abs(dv["MAR"] - 24.5) < 0.05
assert abs(dv["HIN"] - 18.3) < 0.05
print("\n  Malayalam and Gujarati lead; Tamil is lowest despite carrying the")
print("  largest TP inflation, so dependent-vowel isolation alone does not")
print("  determine the fragmentation ordering.")

## References

**This work.**
Anonymous (2026). *Under review.*

**Information Parity (IP).**
Tsvetkov, A., & Kipnis, A. (2024). Information Parity: Measuring and Predicting the
Multilingual Capabilities of Language Models. *Findings of EMNLP 2024*, pp. 7971–7989.

**Tokenization Parity and tokenizer unfairness.**
Petrov, A., La Malfa, E., Torr, P. H. S., & Bibi, A. (2023). Language Model Tokenizers
Introduce Unfairness Between Languages. *NeurIPS 36*.

**COMET.**
Rei, R., Stewart, C., Farinha, A. C., & Lavie, A. (2020). COMET: A Neural Framework for
MT Evaluation. *EMNLP 2020*, pp. 2685–2702. https://aclanthology.org/2020.emnlp-main.213

**IndicMT Eval dataset.**
Sai B., A., Dixit, T., Nagarajan, V., Kunchukuttan, A., Kumar, P., Khapra, M. M., &
Dabre, R. (2023). IndicMT Eval: A Dataset to Meta-Evaluate Machine Translation Metrics
for Indian Languages. *ACL 2023*, pp. 14210–14228. https://aclanthology.org/2023.acl-long.795

**WMT24 Latin-script controls.**
Kocmi, T., et al. (2024). Findings of the WMT24 General Machine Translation Shared Task.
*Proceedings of WMT 2024*, pp. 1–46. https://aclanthology.org/2024.wmt-1.1